# Phase 1: Data Preprocessing (MLflow Audio Pipeline)

---

## What This Notebook Does

This notebook is the **first phase** of our Acoustic Anomaly Detection pipeline, built around the
[DCASE 2024 Challenge Task 2](https://dcase.community/challenge2024/task-first-shot-unsupervised-anomalous-sound-detection-for-machine-condition-monitoring) dataset.

### The Core Problem

Raw audio files are **1-dimensional time-series waveforms** — just a long list of amplitude values.
Neural networks (especially Convolutional Autoencoders) are far more effective when they can
treat data like **2D images**, where spatial patterns become visible.

### Our Solution: Spectrograms as Photographs of Sound

We convert each `.wav` file into a **spectrogram** — a 2D matrix where:

| Axis | Represents | Intuition |
|------|-----------|----------|
| **Y-axis** (rows) | Frequency bands | "Which pitches are present?" |
| **X-axis** (columns) | Time frames | "When do those pitches occur?" |
| **Cell value** | Log-scaled power (dB) | "How loud is that pitch at that moment?" |

Think of it as a **heat-map photograph of sound**. Our future Autoencoder will learn what
"normal" heat-maps look like, and flag anything that deviates as an anomaly.

### Pipeline Steps

1. **Dynamically scan** `./data/raw/` to discover all machine types (fan, pump, valve, etc.)
2. **Load** each `.wav` file at a fixed sample rate of 16 kHz
3. **Extract** spectrograms (Linear or Mel, configurable via Control Panel)
4. **Split** into 85% Training / 15% Validation sets (before scaling — to prevent data leakage)
5. **Normalize** using the selected scaler — fitted ONLY on training data
6. **Save** processed arrays as `.npy` files, scaler as `scaler.save`, and metadata as `prep_meta.json`

> **Why does this matter?** Without this preprocessing step, our Autoencoder would receive
> inconsistent, unnormalized, high-dimensional data and fail to converge during training.

---

## 1. Library Imports

We rely on a focused set of libraries, each chosen for a specific role:

| Library | Purpose |
|---------|--------|
| **`librosa`** | The gold-standard library for audio analysis in Python. Handles loading `.wav` files, computing STFT, Mel spectrograms, and dB conversion. |
| **`numpy`** | Our workhorse for all matrix/array operations. Spectrograms are just NumPy arrays. |
| **`scikit-learn`** | Provides `train_test_split` for data splitting, and `MinMaxScaler` / `StandardScaler` for normalization. |
| **`joblib`** | Serializes our fitted scaler to disk so we can reuse the *exact same* transformation on test data. |
| **`os` / `glob`** | Standard Python modules for filesystem navigation and pattern-based file discovery. |

In [ ]:
import os
import glob
import json

import librosa
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

print("All libraries imported successfully.")
print(f"  librosa  version: {librosa.__version__}")
print(f"  numpy    version: {np.__version__}")

---

## 2. Control Panel

### Why N_FFT = 1024 (Our Optimized "Sweet Spot")

The FFT window size controls the tradeoff between **frequency resolution** and **time resolution**:

| N_FFT | Freq Resolution | Time Resolution | Image Height | Training Speed |
|-------|----------------|----------------|-------------|----------------|
| 2048 | 7.8 Hz/bin | 128 ms/frame | 1025 bins | Slow |
| **1024** | **15.6 Hz/bin** | **64 ms/frame** | **513 bins** | **2× faster** |
| 512 | 31.3 Hz/bin | 32 ms/frame | 257 bins | 4× faster |

With N_FFT=1024, we cut the image size in half (513 vs 1025 bins), which **massively speeds
up training** while still providing 15.6 Hz resolution — more than enough to detect bearing
defects (typically 50–500 Hz bands) and valve leaks (1–4 kHz range).

### Why HOP_LENGTH = 256 (75% Overlap)

HOP_LENGTH determines the step size between successive FFT frames. Setting it to
`N_FFT / 4 = 256` gives us **75% overlap** between adjacent frames, which:
- Prevents information loss at frame boundaries
- Produces smoother spectrograms with better temporal continuity
- Is the standard in audio ML research (e.g., librosa defaults)

### Linear vs. Mel Spectrograms

| Type | How It Works | Best For |
|------|-------------|----------|
| **Linear** (`librosa.stft`) | Every frequency bin is equally spaced. Height = `(N_FFT/2)+1`. | **Mechanical faults** — bearing wear at 4,200 Hz gets its own bin. |
| **Mel** (`melspectrogram`) | Compresses high frequencies into fewer bins. Height = `N_MELS`. | **Quick baselines** — compact, fast to train. |

### StandardScaler vs. MinMaxScaler

| Scaler | Formula | Outlier Behavior |
|--------|---------|------------------|
| **MinMaxScaler** | `(x-min)/(max-min)` → `[0,1]` | A mic pop shifts `max`, squashing ALL data. **Fragile.** |
| **StandardScaler** | `(x-mean)/std` → mean=0, std=1 | Outliers affect mean/std minimally. **Robust.** |

In [ ]:
# ===========================================================================
# CONTROL PANEL — Change these and re-run the notebook
# ===========================================================================
PREP_PARAMS = {
    'SPECTROGRAM_TYPE': 'Mel',     # 'Linear' or 'Mel'
    'SCALER_TYPE':      'MinMax',     # 'Standard' or 'MinMax'
    'N_FFT':            1024,         # FFT window (our optimized sweet spot)
    'HOP_LENGTH':       512,          # 75% overlap (N_FFT / 4)
    'N_MELS':           128,          # Mel bands (only used if Mel)
    'SAMPLE_RATE':      16000,
    'VAL_SPLIT':        0.15,
    'RANDOM_STATE':     42,
}

# Dynamic height calculation
if PREP_PARAMS['SPECTROGRAM_TYPE'] == 'Linear':
    SPEC_HEIGHT = (PREP_PARAMS['N_FFT'] // 2) + 1  # 513 for N_FFT=1024
else:
    SPEC_HEIGHT = PREP_PARAMS['N_MELS']  # 128 for Mel

PREP_PARAMS['SPEC_HEIGHT'] = SPEC_HEIGHT

print("=" * 50)
print("PREP CONTROL PANEL")
print("=" * 50)
for k, v in PREP_PARAMS.items():
    print(f"  {k:<20s} : {v}")

---

## 3. Directory Setup & Dynamic Machine-Type Scanning

### Why Dynamic Scanning?

A common beginner mistake is to **hardcode** machine names:

```python
# BAD — brittle, breaks when you add/remove machines
machines = ['fan', 'pump', 'slider', 'valve']
```

Instead, we **dynamically scan** `./data/raw/` to discover whatever machine folders exist.
This makes our pipeline **portable**, **scalable**, and **robust**.

All paths are **local** to `mlflow_pipeline/` — this folder is a self-contained "Briefcase".

In [ ]:
# Local paths — everything lives inside mlflow_pipeline/
RAW_BASE  = os.path.join(".", "data", "raw")
PROC_BASE = os.path.join(".", "data", "processed")

machine_types = sorted([
    d for d in os.listdir(RAW_BASE)
    if os.path.isdir(os.path.join(RAW_BASE, d))
])

print(f"Discovered {len(machine_types)} machine types:")
for i, m in enumerate(machine_types, 1):
    train_path = os.path.join(RAW_BASE, m, "train")
    n_files = len(glob.glob(os.path.join(train_path, "*.wav"))) if os.path.isdir(train_path) else 0
    print(f"  {i}. {m:<12s}  →  {n_files} training .wav files")

---

## 4. Spectrogram Extraction & Scaling

### The Physics Behind Each Spectrogram

**Linear Spectrogram** (`librosa.stft` → `amplitude_to_db`):
- Computes the Short-Time Fourier Transform with `(N_FFT//2 + 1)` evenly-spaced bins
- A bearing defect at 4,200 Hz gets its own dedicated bin — visible as a bright line
- With N_FFT=1024: each bin = 15.6 Hz wide → 513 bins total

**Mel Spectrogram** (`melspectrogram` → `power_to_db`):
- Applies triangular filter banks that merge high-frequency bins
- That 4,200 Hz defect gets averaged with neighbors — the spike may vanish
- Compact (128 rows), trains faster

### The 3D → 2D → 3D Reshape Trick

Scikit-learn's scalers expect `(n_samples, n_features)` — a flat 2D table.
Our data is `(n_samples, height, width)` — a 3D cube. So we:
1. **Flatten** each spectrogram: `(N, H, W)` → `(N, H*W)`
2. **Fit scaler** on training data ONLY (prevents data leakage)
3. **Reshape** back: `(N, H*W)` → `(N, H, W)`

### The Leakage Guard

```python
# ✅ CORRECT — scaler learns from training data ONLY
scaler.fit_transform(X_train_flat)  # Learn min/max + apply
scaler.transform(X_val_flat)        # Apply same parameters (no re-learning)

# ❌ WRONG — scaler re-learns from validation data
scaler.fit_transform(X_val_flat)    # This would contaminate everything!
```

In [ ]:
SR       = PREP_PARAMS['SAMPLE_RATE']
N_FFT    = PREP_PARAMS['N_FFT']
HOP      = PREP_PARAMS['HOP_LENGTH']
N_MELS   = PREP_PARAMS['N_MELS']
SPEC_T   = PREP_PARAMS['SPECTROGRAM_TYPE']
SCALER_T = PREP_PARAMS['SCALER_TYPE']

for machine in machine_types:
    print(f"\n{'='*70}")
    print(f"PROCESSING: {machine}")
    print(f"{'='*70}")

    proc_dir = os.path.join(PROC_BASE, machine)
    os.makedirs(proc_dir, exist_ok=True)

    wav_files = sorted(glob.glob(os.path.join(RAW_BASE, machine, "train", "*.wav")))
    if not wav_files:
        print(f"  [WARNING] No .wav files found. Skipping.")
        continue
    print(f"  Found {len(wav_files)} .wav files")

    # ----- Spectrogram extraction -----
    specs = []
    for i, fp in enumerate(wav_files):
        y, sr = librosa.load(fp, sr=SR)

        if SPEC_T == 'Linear':
            stft = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP))
            spec_db = librosa.amplitude_to_db(stft, ref=np.max)
        else:
            mel = librosa.feature.melspectrogram(
                y=y, sr=sr, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
            spec_db = librosa.power_to_db(mel, ref=np.max)

        specs.append(spec_db)
        if (i + 1) % 200 == 0 or (i + 1) == len(wav_files):
            print(f"    Processed {i+1}/{len(wav_files)} files...")

    X_all = np.array(specs)
    print(f"  Combined array shape: {X_all.shape}")
    print(f"  Value range: [{X_all.min():.2f}, {X_all.max():.2f}] dB")

    # ----- Train / Val split (BEFORE scaling) -----
    X_train, X_val = train_test_split(
        X_all, test_size=PREP_PARAMS['VAL_SPLIT'],
        random_state=PREP_PARAMS['RANDOM_STATE'])

    print(f"\n  Train/Val Split (seed={PREP_PARAMS['RANDOM_STATE']}):")
    print(f"    X_train : {X_train.shape}  ({X_train.shape[0]} samples)")
    print(f"    X_val   : {X_val.shape}  ({X_val.shape[0]} samples)")

    # ----- 3D -> 2D -> Scale -> 3D -----
    N_tr, H, W = X_train.shape
    N_va = X_val.shape[0]

    if SCALER_T == 'Standard':
        scaler = StandardScaler()
    else:
        scaler = MinMaxScaler(feature_range=(0, 1))

    X_tr_s = scaler.fit_transform(X_train.reshape(-1, 1)).reshape(N_tr, H, W)
    X_va_s = scaler.transform(X_val.reshape(-1, 1)).reshape(N_va, H, W)

    print(f"\n  After {SCALER_T}Scaler:")
    print(f"    X_train range: [{X_tr_s.min():.4f}, {X_tr_s.max():.4f}]")
    print(f"    X_val   range: [{X_va_s.min():.4f}, {X_va_s.max():.4f}]")

    # ----- Save -----
    np.save(os.path.join(proc_dir, "X_train.npy"), X_tr_s)
    np.save(os.path.join(proc_dir, "X_val.npy"), X_va_s)
    joblib.dump(scaler, os.path.join(proc_dir, "scaler.save"))

    meta = {**PREP_PARAMS, 'SPEC_HEIGHT': H, 'SPEC_WIDTH': W}
    with open(os.path.join(proc_dir, "prep_meta.json"), 'w') as f:
        json.dump(meta, f, indent=2)

    print(f"\n  Saved to {proc_dir}:")
    print(f"    ✓ X_train.npy   — {X_tr_s.shape}")
    print(f"    ✓ X_val.npy     — {X_va_s.shape}")
    print(f"    ✓ scaler.save   — {SCALER_T}Scaler fitted on training data")
    print(f"    ✓ prep_meta.json")

---

## 5. Verification — Quick Sanity Check

Let's verify that our saved files can be loaded and contain sensible data.

In [ ]:
print("Verification: Reloading saved files...\n")

for machine in machine_types:
    processed_dir = os.path.join(PROC_BASE, machine)
    train_file  = os.path.join(processed_dir, "X_train.npy")
    val_file    = os.path.join(processed_dir, "X_val.npy")
    scaler_file = os.path.join(processed_dir, "scaler.save")

    if not all(os.path.exists(f) for f in [train_file, val_file, scaler_file]):
        print(f"  [{machine}] ⚠ Missing files"); continue

    X_tr = np.load(train_file)
    X_va = np.load(val_file)
    sc   = joblib.load(scaler_file)

    print(f"  [{machine}] ✓ PASS")
    print(f"    X_train : {X_tr.shape}  range=[{X_tr.min():.4f}, {X_tr.max():.4f}]")
    print(f"    X_val   : {X_va.shape}  range=[{X_va.min():.4f}, {X_va.max():.4f}]")
    print(f"    Scaler  : {type(sc).__name__}")
    print()

---

## Summary & Next Steps

We have successfully:

1. ✅ Loaded raw `.wav` audio files
2. ✅ Extracted spectrograms (Linear or Mel, configurable)
3. ✅ Split data into train/validation sets (before scaling)
4. ✅ Scaled using the selected scaler (fitted on train only)
5. ✅ Saved everything as `.npy` + `scaler.save` + `prep_meta.json`

**Next:** Open `02_mlflow_train.ipynb` to build and train the CNN Autoencoder.